**Code Credit: Hugging Face**

**Dataset Credit: https://twitter.com/Dorialexander/status/1681671177696161794 **

## Finetune Llama-2-7b on a Google colab

Welcome to this Google Colab notebook that shows how to fine-tune the recent Llama-2-7b model on a single Google colab and turn it into a chatbot

We will leverage PEFT library from Hugging Face ecosystem, as well as QLoRA for more memory efficient finetuning

## Setup

Run the cells below to setup and install the required libraries. For our experiment we will need `accelerate`, `peft`, `transformers`, `datasets` and TRL to leverage the recent [`SFTTrainer`](https://huggingface.co/docs/trl/main/en/sft_trainer). We will use `bitsandbytes` to [quantize the base model into 4bit](https://huggingface.co/blog/4bit-transformers-bitsandbytes). We will also install `einops` as it is a requirement to load Falcon models.

In [ ]:
!pip install -q -U trl transformers accelerate git+https://github.com/huggingface/peft.git
!pip install -q datasets bitsandbytes einops wandb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Dataset



In [ ]:
from datasets import load_dataset
from smart_open import smart_open
import pandas as pd

dataset = load_dataset("b-mc2/sql-create-context")

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'context', 'answer'],
        num_rows: 78577
    })
})

In [ ]:
dataset['train'].features

{'question': Value(dtype='string', id=None),
 'context': Value(dtype='string', id=None),
 'answer': Value(dtype='string', id=None)}

In [ ]:
def format_prompt(r):
    return f'''###Context: {r["context"]}\n###Question: {r["question"]}\n###Answer: {r["answer"]}'''

def convert_dataset(ds):
    prompts = [format_prompt(i) for i in ds if i['context'] != '']
    # labels = [format_query(i) for i in ds if i['context'] != '']
    df = pd.DataFrame.from_dict({'text': prompts})
    return df

In [ ]:
dataset_df = convert_dataset(dataset["train"])

In [ ]:
dataset_df

,text
0,###Context: CREATE TABLE head (age INTEGER)\n#...
1,"###Context: CREATE TABLE head (name VARCHAR, b..."
2,###Context: CREATE TABLE department (creation ...
3,###Context: CREATE TABLE department (budget_in...
4,###Context: CREATE TABLE department (num_emplo...
...,...
78572,###Context: CREATE TABLE table_name_35 (time V...
78573,###Context: CREATE TABLE table_name_83 (ground...
78574,###Context: CREATE TABLE table_name_60 (compet...
78575,###Context: CREATE TABLE table_name_34 (decile...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Errno 2] No such file or directory: 'drive/MyDrive'
/content/drive/MyDrive


In [ ]:
etli_data_model = pd.read_csv("create_queries_df_v2.csv")
etli_data_model.drop("Unnamed: 0", axis = 1, inplace = True)

In [ ]:
etli_data_model["text"] = etli_data_model.apply(lambda x: (f'''###Context: {x["context"]}\n###Question: {x["question"]}\n###Answer: {x["answer"]}'''),axis=1)

In [ ]:
print(etli_data_model.text[1])

###Context: CREATE EXTERNAL TABLE s_agnt_dtl( _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hsh_ky_agnt_cd string , rcrd_hsh_id string , agnt_sqnc_nmbr string , prty_id string , prty_ctgry_cd string , vrsn_id string , agnt_applctn_nmbr string , agnt_applctn_dt string , trnsctn_stts_cd string , agnt_applctn_stts_dt string , prvs_entty_stts_dtl string , rjctn_rsn_dscrptn string , src_cd_gnrtn_dt timestamp , src_systm_stts_cd string , prnt_agnt_sqnc_nmbr string , cntry_cd string , src_cd_end_dt timestamp , src_systm_crtd_dt timestamp , src_systm_updtd_dt timestamp , prtl_rfrnc_nmbr string , agnt_cmpny_cd string , agnt_cstmr_id string , agnt_cd_vrfcn_flg string , dsgntn_nm string , emply_cd string , chnnl_cd string , chnnl_nm string , sb_chnnl_cd string , sb_chnnl_nm string , agnt_sprvsr_cd string , agnt_sprvsr_nm string , typ_nm string , agnt_brth_dt timestamp , jb_ttl_cd string , ulip_trng

In [ ]:
dataset_df = dataset_df.text.append(etli_data_model.text).reset_index().drop("index", axis = 1)

<ipython-input-39-f15dc2a87957>:1: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dataset_df = dataset_df.text.append(etli_data_model.text).reset_index().drop("index", axis = 1)


In [ ]:
dataset_df

,text
0,###Context: CREATE TABLE head (age INTEGER)\n#...
1,"###Context: CREATE TABLE head (name VARCHAR, b..."
2,###Context: CREATE TABLE department (creation ...
3,###Context: CREATE TABLE department (budget_in...
4,###Context: CREATE TABLE department (num_emplo...
...,...
78675,###Context: CREATE TABLE v_stg_s_plcy_addtnl_d...
78676,###Context: CREATE TABLE raw_s_plcy_loan( plcy...
78677,###Context: CREATE TABLE raw_l_applctn_chnnl( ...
78678,###Context: CREATE EXTERNAL TABLE s_plcy_cvrg(...


In [ ]:
78782-3000

75782

In [ ]:
import pandas as pd
from datasets import DatasetDict, Dataset

In [ ]:
dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(dataset_df[75782:])
})

In [ ]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2898
    })
})

In [ ]:
dataset_dict['train']

Dataset({
    features: ['text'],
    num_rows: 2898
})

In [ ]:
# from datasets import load_dataset

# #dataset_name = "timdettmers/openassistant-guanaco" ###Human ,.,,,,,, ###Assistant

# dataset_name = 'AlexanderDoria/novel17_test' #french novels
# dataset = load_dataset(dataset_name, split="train")

## Loading the model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

model_name = "TinyPixel/Llama-2-7B-bf16-sharded"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

Let's also load the tokenizer below

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from peft import LoraConfig, get_peft_model

lora_alpha = 16
lora_dropout = 0.1
lora_r = 64

peft_config = LoraConfig(
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    r = lora_r,
    bias = "none",
    task_type = "CAUSAL_LM"
)

## Loading the trainer

Here we will use the [`SFTTrainer` from TRL library](https://huggingface.co/docs/trl/main/en/sft_trainer) that gives a wrapper around transformers `Trainer` to easily fine-tune models on instruction based datasets using PEFT adapters. Let's first load the training arguments below.

In [ ]:
from transformers import TrainingArguments

output_dir = "./results"
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
optim = "paged_adamw_32bit"
save_steps = 100
logging_steps = 10
learning_rate = 2e-4
max_grad_norm = 0.3
max_steps = 100
warmup_ratio = 0.03
lr_scheduler_type = "constant"

training_arguments = TrainingArguments(
    output_dir = output_dir,
    per_device_train_batch_size = per_device_train_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    optim = optim,
    save_steps = save_steps,
    logging_steps = logging_steps,
    learning_rate = learning_rate,
    fp16 = True,
    max_grad_norm = max_grad_norm,
    max_steps = max_steps,
    warmup_ratio = warmup_ratio,
    group_by_length = True,
    lr_scheduler_type = lr_scheduler_type,
)

Then finally pass everthing to the trainer

In [ ]:
from trl import SFTTrainer

max_seq_length = 512

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset_dict['train'],
    peft_config = peft_config,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = training_arguments,
)

/usr/local/lib/python3.10/dist-packages/peft/utils/other.py:133: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(


Map:   0%|          | 0/2898 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:207: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


We will also pre-process the model by upcasting the layer norms in float 32 for more stable training

In [ ]:
for name, module in trainer.model.named_modules():
    if "norm" in name:
        module = module.to(torch.float32)

## Train the model

Now let's train the model! Simply call `trainer.train()`

In [ ]:
# 12af4bb8155692bd37605c51c4462ad748fca9ae

In [ ]:
import wandb
wandb.login(key="12af4bb8155692bd37605c51c4462ad748fca9ae")
wandb.init()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [ ]:
trainer.train()

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
10,7.453700
20,3.709100
30,1.640500
40,1.456200
50,2.865900
60,2.705700
70,1.667900
80,1.722100
90,1.859000
100,34.176700


TrainOutput(global_step=100, training_loss=5.925678434371949, metrics={'train_runtime': 920.9835, 'train_samples_per_second': 1.737, 'train_steps_per_second': 0.109, 'total_flos': 3135863355310080.0, 'train_loss': 5.925678434371949, 'epoch': 0.55})

In [ ]:
model_to_save = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model  # Take care of distributed/parallel training
model_to_save.save_pretrained("outputs")

In [ ]:
lora_config = LoraConfig.from_pretrained('outputs')
model = get_peft_model(model, lora_config)

In [ ]:
# dataset_df['text'].tail(60)

In [ ]:
dataset_df.shape

(78680, 1)

In [ ]:
dataset_df['text'][78635]

'###Context: CREATE TABLE v_stg_s_cstmr_incm( cstmr_id string, rcrd_src_nm string, cstmr_incm_effctv_dt date, cstmr_incm_amnt decimal(13,0), cstmr_othr_incm_amnt decimal(13,0), cstmr_incm_cd string, cstmr_othr_incm_cd string, cstmr_nt_wrth_amnt decimal(15,0), cstmr_incm_prf_cd string, cstmr_hshld_incm_amnt decimal(15,0), ld_dt_tm timestamp, hsh_ky_cstmr_cd string, rcrd_hsh_id string)\n###Question: Create a table named Satellite Customer Income as v_stg_s_cstmr_incm and the columns names as customer Identifier (BK) string, Record Source Name string, Customer Income Effective Date date, Customer Income Amount decimal(13,0), Customer Other Income Amount decimal(13,0), Customer Income Code string, Customer Other Income Code string, Customer Net Worth Amount decimal(15,0), Customer Income Proof Code string, Customer Household Income Amount decimal(15,0), Load Date Time timestamp, Hash Key Customer Code string, Record Hash Identifier string)\n###Answer: CREATE TABLE v_stg_s_cstmr_incm( cstmr

In [ ]:
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string.,
answer: CREATE TABLE v_stg_l_agnt_chnnl( rcrd_src_nm string, ld_dt_tm timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string

example 2:-
question: Create a table named  h_agnt with the columns names: _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , Agent Identifier string , Load Date Time timestamp , Record Source Name string
answer: CREATE EXTERNAL TABLE h_agnt( _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , agnt_id string , ld_dt_tm timestamp , rcrd_src_nm string

now gwnerate sql for this question:- SELECT customer name where customer income amount is greater then 50000 """

# select all rcrd_src_nm where ld_dt_tm is after august 2021
# select all l_hk_agnt_chnnl_cd where Load Date Time is after august 2021
# Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string
device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 50)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string.,
answer: CREATE TABLE v_stg_l_agnt_chnnl( rcrd_src_nm string, ld_dt_tm timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string

example 2:-
question: Create a table named  h_agnt with the columns names: _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , Agent Identifier string , Load Date Time timestamp , Record Source Name string
answer: CREATE EXTERNAL TABLE h_agnt( _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_

In [ ]:
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string.,
answer: CREATE TABLE v_stg_l_agnt_chnnl( rcrd_src_nm string, ld_dt_tm timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string

example 2:-
question: Create a table named  h_agnt with the columns names: _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , Agent Identifier string , Load Date Time timestamp , Record Source Name string
answer: CREATE EXTERNAL TABLE h_agnt( _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , agnt_id string , ld_dt_tm timestamp , rcrd_src_nm string

now gwnerate sql for this question:- SELECT Hub Agent Identifier WHERE Load Date Time is after 2021"""

# select all rcrd_src_nm where ld_dt_tm is after august 2021
# select all l_hk_agnt_chnnl_cd where Load Date Time is after august 2021
# Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string
device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 50)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string.,
answer: CREATE TABLE v_stg_l_agnt_chnnl( rcrd_src_nm string, ld_dt_tm timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string

example 2:-
question: Create a table named  h_agnt with the columns names: _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , Agent Identifier string , Load Date Time timestamp , Record Source Name string
answer: CREATE EXTERNAL TABLE h_agnt( _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_

In [ ]:
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now gwnerate sql for this question:- List the name, born state and age of the heads of departments ordered by age"""

device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 100)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now gwnerate sql for this question:- List the name, born state and age of the heads of departments ordered by age in descending order.

Answer:-

SELECT DISTINCT Name, Born_State, Age FROM Employee ORDER BY Age DESC

Answer:-

SELECT DISTINCT Name, Born_State, Age FROM Employee ORDER BY Age DESC

Answer:-

SELECT DISTINCT Name, Born_State, Age FROM Employee ORDER BY Age DESC

Answer:-

SELECT DISTINCT Name, Born_State, A

In [ ]:
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

example 3:-
question: Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string.,
answer: CREATE TABLE v_stg_l_agnt_chnnl( rcrd_src_nm string, ld_dt_tm timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string

example 4:-
question: Create a table named  h_agnt with the columns names: _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , Agent Identifier string , Load Date Time timestamp , Record Source Name string
answer: CREATE EXTERNAL TABLE h_agnt( _hoodie_commit_time string , _hoodie_commit_seqno string , _hoodie_record_key string , _hoodie_partition_path string , _hoodie_file_name string , hk_agnt_cd string , agnt_id string

now gwnerate sql for this question:- Create a table named raw_h_cstmr with the columns names: Customer Identifier string, Record Source Name string"""

device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 100)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

example 3:-
question: Create a table named  v_stg_l_agnt_chnnl with the columns names: Record Source Name string, Load Date Time timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string.,
answer: CREATE TABLE v_stg_l_agnt_chnnl( rcrd_src_nm string, ld_dt_tm timestamp, l_hk_agnt_chnnl_cd string, h_agnt_cd string, h_chnnl_cd string

example 4:-
question: Create a table named  h_agnt with the columns names:

In [ ]:
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now gwnerate sql for this question:- Create a table named  raw_l_applctn_chnnl with the columns names: h_applctn_cd string, h_chnnl_cd string, Record Source Name string"""

device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 100)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now gwnerate sql for this question:- Create a table named  raw_l_applctn_chnnl with the columns names: h_applctn_cd string, h_chnnl_cd string, Record Source Name string, Record Source Type string, Record Source ID string, Record Source ID Type string, Record Source ID Number string, Record Source ID Number Type string, Record Source ID Number Length string, Record Source ID Number Format string, Record Source ID Number F

In [ ]:
# text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

# here is some example:-

# example 1:-
# question: Please show the themes of competitions with host cities having populations larger than 1000.,
# answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

# example 2:-
# question: Please show the different statuses of cities and the average population of cities with each status.,
# answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

# now gwnerate sql for this question:- Find all the customer information in state NY"""

In [ ]:
# 2
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now gwnerate sql for this question:- List the creation year, name and budget of each department"""

In [ ]:
# 3
# text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

# here is some example:-

# example 1:-
# question: Please show the themes of competitions with host cities having populations larger than 1000.,
# answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

# example 2:-
# question: Please show the different statuses of cities and the average population of cities with each status.,
# answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

# now gwnerate sql for this question:- What are the maximum and minimum budget of the departments?"""

In [ ]:
# 90
# text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

# here is some example:-

# example 1:-
# question: Please show the themes of competitions with host cities having populations larger than 1000.,
# answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

# example 2:-
# question: Please show the different statuses of cities and the average population of cities with each status.,
# answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

# now gwnerate sql for this question:- Which trip started from the station with the largest dock count? Give me the trip id"""

In [ ]:
# text = "###Question: List all the cities in a decreasing order of each city's stations' highest latitude."
# text = "###Question: List the name, born state and age of the heads of departments ordered by age."
# text = "###Question: How many heads of the departments are older than 56 ?"
# text = "###Context: CREATE TABLE head (age INTEGER)\n###Question: How many heads of the departments are older than 56 ?"
# text = "###Question: Find all the customer information in state NY"
device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 100)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here is some example:-

example 1:- 
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:- 
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now gwnerate sql for this question:- List the creation year, name and budget of each department in the company.,
answer: SELECT YEAR(Created_Date), Name, Budget FROM Department

now gwnerate sql for this question:- List the creation year, name and budget of each department in the company.,
answer: SELECT YEAR(Created_Date), Name, Budget FROM Department

now gwnerate sql for this question:- List the creation year, name 

In [ ]:
# SELECT
#   T1.id
# FROM
#   trip AS T1
#     JOIN station AS T2 ON T1.start_station_id = T2.id
# ORDER BY
#   T2.dock_count
# DESC
#   LIMIT 1

In [ ]:
# SELECT
#   T1.Trip_ID,
#   T1.Dock_Count
# FROM
#   Trip AS T1
#     JOIN Station AS T2 ON T1.Station_ID = T2.ID
# WHERE
#   T2.Dock_Count > 10

In [ ]:
dataset_df.shape

(78680, 1)

In [ ]:
dataset_df.text[77681]

'###Context: CREATE TABLE table_name_15 (winner VARCHAR, year VARCHAR, country VARCHAR, score VARCHAR)\n###Question: What is the total number of 1st prize ($) that has a United States country with a score lower than 271 and in a year after 1964 with a winner of Bobby Mitchell?\n###Answer: SELECT COUNT(1 AS st_prize___) AS $__ FROM table_name_15 WHERE country = "united states" AND score < 271 AND year > 1964 AND winner = "bobby mitchell"'

In [ ]:
text = """you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here are some examples:-

example 1:-
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:-
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now generate sql for this question:- #Question: What is the total number of 1st prize ($) that has a United States country with a score lower than 271 and in a year after 1964 with a winner of Bobby Mitchell?"""

device = "cuda:0"

inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 100)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:31: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn("None of the inputs have requires_grad=True. Gradients will be None")


you are a smart text to sql developer, i will give you text and you must have to generate sql code for that question,

here are some examples:-

example 1:- 
question: Please show the themes of competitions with host cities having populations larger than 1000.,
answer: SELECT T2.Theme FROM city AS T1 JOIN farm_competition AS T2 ON T1.City_ID = T2.Host_city_ID WHERE T1.Population > 1000

example 2:- 
question: Please show the different statuses of cities and the average population of cities with each status.,
answer: SELECT Status, AVG(Population) FROM city GROUP BY Status

now generate sql for this question:- #Question: What is the total number of 1st prize ($) that has a United States country with a score lower than 271 and in a year after 1964 with a winner of Bobby Mitchell?

Answer: SELECT COUNT(1) FROM (SELECT DISTINCT Country_ID, Year_ID, Winner_ID FROM farm_competition WHERE Country_ID = 'United States' AND Year_ID < 1965 AND Winner_ID = 'Bobby Mitchell')

Comment: Please add so

In [ ]:
#correct
SELECT COUNT(1 AS st_prize___) AS $__
FROM table_name_15
WHERE country = "united states" AND score < 271 AND year > 1964 AND winner = "bobby mitchell

In [ ]:
SELECT COUNT(1)
FROM (SELECT DISTINCT Country_ID, Year_ID, Winner_ID FROM farm_competition
      WHERE Country_ID = 'United States' AND Year_ID < 1965 AND Winner_ID = 'Bobby Mitchell')


In [ ]:
# select all queries from data with table name "table_name_15" and append with main traning data

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model.push_to_hub("llama2-finetunined-on-text-to-sql-3000")

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM

config = PeftConfig.from_pretrained("Dukul/llama2-finetunined-on-text-to-sql-3000")
model = AutoModelForCausalLM.from_pretrained("TinyPixel/Llama-2-7B-bf16-sharded")
model = PeftModel.from_pretrained(model, "Dukul/llama2-finetunined-on-text-to-sql-3000")

In [ ]:
inputs = tokenizer(text, return_tensors = "pt").to(device)
outputs = model.generate(**inputs, max_new_tokens = 100)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))